# L3 — Train

Latin → English, from scratch. No pretraining: at ~94k pairs the parallel data
teaches the task directly, which is what the paper did at larger scale.

**The number to beat is 0.31** — the copy baseline, i.e. emitting the Latin
unchanged. Unlike the Shakespeare task, where the baseline was 19.22 and
unbeatable, here almost any real translation clears it. That is the point of
the pivot: a score can finally mean what it appears to mean.

In [1]:
import os, subprocess, sys

REPO = "https://github.com/Amay-M-Nair/AttentionMech.git"

def has_src(d):
    return os.path.isdir(os.path.join(d, "src"))

# On Kaggle the notebook starts in /kaggle/working with no repo. Clone it, or
# pull if this session already cloned once.
if not any(has_src(d) for d in (os.getcwd(), os.path.dirname(os.getcwd()))):
    target = os.path.join(os.getcwd(), "AttentionMech")
    if os.path.isdir(os.path.join(target, ".git")):
        subprocess.run(["git", "-C", target, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO, target], check=True)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "sentencepiece", "sacrebleu"], check=True)

print("cwd:", os.getcwd())

cwd: c:\Games\Codes\Python\Projects\2.transformer-from-scratch\notebooks


In [2]:
# Walk up from wherever the kernel started until a directory with src/ turns up.
# Self-contained on purpose: this cell must not depend on the one above it
# having run, and a notebook's cwd is not reliably its own directory.
import os, sys

ROOT = os.getcwd()
while not os.path.isdir(os.path.join(ROOT, "src")) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "src")):
    raise RuntimeError(f"no src/ above {os.getcwd()} - run the setup cell first")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print("root:", ROOT)

import json
import os, sys, time
from dataclasses import asdict

import matplotlib.pyplot as plt
import torch

from src.config import TransformerConfig, get_device
from src.dataset import TranslationDataset, load_split, make_dataloader
from src.evaluate import bleu, copy_baseline
from src.inference import translate_corpus
from src.spm_tokenizer import SPMTokenizer
from src.train import fit, load_checkpoint, overfit_batch
from src.transformer import build_model

%matplotlib inline
DATA = os.path.join(ROOT, "data")
CHECKPOINT = os.path.join(ROOT, "checkpoints", "latin.pt")
OUTPUTS = os.path.join(ROOT, "outputs")
os.makedirs(OUTPUTS, exist_ok=True)

device = get_device()
torch.manual_seed(0)

tokenizer = SPMTokenizer(os.path.join(DATA, "spm8k.model"))
train_la, train_en = load_split(DATA, "train")
valid_la, valid_en = load_split(DATA, "valid")
test_la, test_en = load_split(DATA, "test")

valid_baseline = copy_baseline(valid_la, valid_en)
test_baseline = copy_baseline(test_la, test_en)

print("device:", device, "| vocab:", len(tokenizer))
print(f"train {len(train_la):,} | valid {len(valid_la):,} | test {len(test_la):,}")
print(f"copy baseline   valid {valid_baseline:.2f}   test {test_baseline:.2f}")

root: c:\Games\Codes\Python\Projects\2.transformer-from-scratch
device: cuda | vocab: 8000
train 94,099 | valid 3,003 | test 3,003
copy baseline   valid 0.27   test 0.31


## Config

Every number here follows from something measured:

| Setting | Value | Why |
|---|---|---|
| vocab | 8,000 | L2 — small on purpose; a 32k vocab over 94k pairs is itself a cause of overfitting |
| `max_len` | 128 | L2 — covers 97.4% of pairs untruncated; 192 buys 2 points for 50% more compute |
| d_model / layers | 384, 4+4 | ~20M params. The 37M model memorised 18k Shakespeare pairs in 3 epochs |
| dropout | 0.3 | kept. Run 1 showed no overfitting across 40 epochs, so this was never the binding constraint and lowering it is untested |
| lr / warmup | 5e-4 / 4000 | **run 1 used 400.** Inverse-sqrt decay took the rate to 2.9e-5 by epoch 40 with BLEU still climbing; 4000 holds it ~3.2x higher throughout |
| epochs | 60 | run 1 was still improving at 40 - every epoch marked best, so patience 8 never fired |

In [3]:
MAX_LEN, BATCH = 128, 32

config = TransformerConfig(
    vocab_size=len(tokenizer),
    d_model=384, num_heads=8, num_layers=4, d_ff=1536,
    dropout=0.3, max_len=512,
)

model = build_model(config).to(device)
for name, count in model.count_parameters().items():
    print(f"  {name:<12} {count:>12,}")

  src_embed       3,072,000
  encoder         7,092,480
  decoder         9,454,848
  TOTAL          19,619,328


In [ ]:
train_loader = make_dataloader(
    TranslationDataset(train_la, train_en, tokenizer, max_len=MAX_LEN), batch_size=BATCH)
valid_loader = make_dataloader(
    TranslationDataset(valid_la, valid_en, tokenizer, max_len=MAX_LEN),
    batch_size=BATCH, shuffle=False)

src, tgt_in, tgt_out = next(iter(train_loader))
print("src   ", tuple(src.shape), "|", tokenizer.decode(src[0])[:80])
print("target", tuple(tgt_out.shape), "|", tokenizer.decode(tgt_out[0])[:80])
print(f"\n{len(train_loader):,} batches per epoch")

## Gate — overfit one batch

The check that has caught every wiring bug in this project. If the model cannot
memorise 32 pairs, something is wrong and a two-hour run would only discover it
slowly. Dropout off so nothing puts a floor under the loss.

In [ ]:
probe_config = TransformerConfig(
    vocab_size=len(tokenizer), d_model=384, num_heads=8, num_layers=4,
    d_ff=1536, dropout=0.0, max_len=512)
probe = build_model(probe_config).to(device)

batch = tuple(t.to(device) for t in next(iter(train_loader)))
h = overfit_batch(probe, batch, steps=400, log_every=100)
print(f"\nloss {h['loss'][0]:.3f} -> {h['loss'][-1]:.4f}   acc {h['acc'][-1]:.1%}")
print("GATE:", "PASS" if h["acc"][-1] > 0.85 else "FAIL - do not start the long run")

del probe
torch.cuda.empty_cache()

## Train

Validation scores with beam 4, so checkpoint selection optimises the metric
that gets reported. Early stopping on patience 8.

In [ ]:
started = time.time()

history = fit(
    model, config, train_loader, valid_loader,
    valid_la, valid_en, tokenizer, device,
    epochs=60, patience=8, lr=5e-4, warmup=4000,
    label_smoothing=0.1, checkpoint_path=CHECKPOINT, baseline=valid_baseline,
    eval_beam=4, eval_alpha=1.0,
    eval_batch=24, eval_src_max_len=MAX_LEN, eval_max_len=200,
    eval_limit=1000, amp=True,
)

print(f"\ntrained in {(time.time() - started) / 60:.1f} min")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["epoch"], history["train_loss"], label="train")
axes[0].plot(history["epoch"], history["valid_loss"], label="valid")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_title("Loss")

axes[1].plot(history["epoch"], [a*100 for a in history["train_acc"]], label="train")
axes[1].plot(history["epoch"], [a*100 for a in history["valid_acc"]], label="valid")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("token accuracy %")
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_title("Accuracy (gap = overfitting)")

axes[2].plot(history["epoch"], history["bleu"], color="tab:green", label="model")
axes[2].axhline(valid_baseline, color="tab:red", ls="--", label=f"copy baseline {valid_baseline:.2f}")
axes[2].set_xlabel("epoch"); axes[2].set_ylabel("BLEU")
axes[2].legend(); axes[2].grid(alpha=0.3); axes[2].set_title("Validation BLEU")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

best = max(history["bleu"])
gap = (history["train_acc"][-1] - history["valid_acc"][-1]) * 100
print(f"best validation BLEU {best:.2f}   (baseline {valid_baseline:.2f})")
print(f"final train/valid accuracy gap {gap:.1f} points")

# The curves are the evidence for the next run's config, so keep the numbers
# and not just the picture.
with open(os.path.join(OUTPUTS, "train_history.json"), "w") as f:
    json.dump({"config": asdict(config), "valid_baseline": valid_baseline,
               "best_bleu": best, "history": history}, f, indent=1)
print("wrote", OUTPUTS)

## Sanity check

Read the output before trusting the number.

In [ ]:
best_model, meta = load_checkpoint(CHECKPOINT, device=device)
print(f"loaded epoch {meta['epoch']}, valid BLEU {meta['score']:.2f}\n")

sample = valid_la[:8]
predictions = translate_corpus(best_model, sample, tokenizer, device,
                               batch_size=16, beam_size=4, length_penalty=1.0)

for i in range(8):
    print("LA  :", sample[i][:100])
    print("PRED:", predictions[i][:100])
    print("REF :", valid_en[i][:100])
    print()

## Done

`checkpoints/latin.pt` — the first model in this project measured against a
baseline it can actually beat.

Next: **L4** — tune decoding on validation, score test once, and report BLEU and
chrF beside the 0.31 baseline, split by sentence length and by source work.